In [125]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import os
import pprint

In [126]:
raw_kmt_dataset = Path("raw_kmt_dataset")
json_file_list = os.listdir(raw_kmt_dataset)
json_file_list.sort()
json_file_list

['raw_kmt_user_0001.json',
 'raw_kmt_user_0002.json',
 'raw_kmt_user_0003.json',
 'raw_kmt_user_0004.json',
 'raw_kmt_user_0005.json',
 'raw_kmt_user_0006.json',
 'raw_kmt_user_0007.json',
 'raw_kmt_user_0008.json',
 'raw_kmt_user_0009.json',
 'raw_kmt_user_0010.json',
 'raw_kmt_user_0011.json',
 'raw_kmt_user_0012.json',
 'raw_kmt_user_0013.json',
 'raw_kmt_user_0014.json',
 'raw_kmt_user_0015.json',
 'raw_kmt_user_0016.json',
 'raw_kmt_user_0017.json',
 'raw_kmt_user_0018.json',
 'raw_kmt_user_0019.json',
 'raw_kmt_user_0020.json',
 'raw_kmt_user_0021.json',
 'raw_kmt_user_0022.json',
 'raw_kmt_user_0023.json',
 'raw_kmt_user_0024.json',
 'raw_kmt_user_0025.json',
 'raw_kmt_user_0026.json',
 'raw_kmt_user_0027.json',
 'raw_kmt_user_0028.json',
 'raw_kmt_user_0029.json',
 'raw_kmt_user_0030.json',
 'raw_kmt_user_0031.json',
 'raw_kmt_user_0032.json',
 'raw_kmt_user_0033.json',
 'raw_kmt_user_0034.json',
 'raw_kmt_user_0035.json',
 'raw_kmt_user_0036.json',
 'raw_kmt_user_0037.json',
 

In [127]:
with open(raw_kmt_dataset / json_file_list[0], 'r') as f:
    user1 = json.load(f)

print(user1.keys())

dict_keys(['details', 'true_data', 'false_data'])


In [128]:
print(user1["details"])
print(user1["false_data"].keys())
print(user1["true_data"].keys())

{'ID': 'CDID0001', 'Provider': 'Discover', 'Name': 'Mrs Ellie Miller', 'Card Number': 4422891459068728, 'CVC': 646, 'Expiry': '09/23'}
dict_keys(['test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'test_6', 'test_7', 'test_8', 'test_9', 'test_10'])
dict_keys(['test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'test_6', 'test_7', 'test_8', 'test_9', 'test_10'])


In [129]:
print(user1["true_data"]["test_1"].keys())

dict_keys(['key_events', 'mouse_events'])


In [130]:
user1_key = pd.DataFrame(user1["true_data"]["test_1"]["key_events"])

display(user1_key)
display(user1_key.describe(include="all"))
display(user1_key.dtypes)

,Key,Event,Input Box,Text Changed,Timestamp,Epoch
0,shift,pressed,Name,True,2022-02-06 18:57:09.981740,1644173829.9817405
1,m,pressed,Name,True,2022-02-06 18:57:10.121021,1644173830.1210213
2,shift,released,Name,True,2022-02-06 18:57:10.169470,1644173830.1694703
3,m,released,Name,True,2022-02-06 18:57:10.218179,1644173830.21818
4,r,pressed,Name,True,2022-02-06 18:57:10.293582,1644173830.293582
...,...,...,...,...,...,...
113,tab,released,Exp y,True,2022-02-06 18:57:26.707177,1644173846.7071774
114,2,pressed,Exp y,True,2022-02-06 18:57:26.834916,1644173846.8349168
115,2,released,Exp y,True,2022-02-06 18:57:26.895162,1644173846.8951626
116,3,pressed,Exp y,True,2022-02-06 18:57:26.958561,1644173846.9585614


,Key,Event,Input Box,Text Changed,Timestamp,Epoch
count,118,118,118,118,118,118
unique,21,2,5,2,116,116
top,backspace,released,Name,True,2022-02-06 18:57:10.756190,1644173830.7561908
freq,14,61,64,112,2,2


Key              str
Event            str
Input Box        str
Text Changed    bool
Timestamp        str
Epoch            str
dtype: object

## First glance
`Text Changed` and `Epoch` features can be dropped as they neither posses any directly useful discriminatory value nor they can be of any help in feature engineering.

Consider dropping `Shift` key as well as other possible modifier keys as they are pressed and held for a longer time.


>**NOTE**
>
>Differentiating between using `Tab` and/or `left-click` for moving between the fields on the form may have discriminatory power.

Let's look at all files concatenated to check for other possible discriminating features.

In [187]:
dfs = []
for file in json_file_list:
    with open(raw_kmt_dataset / file, 'r') as f:
        user_data = json.load(f)
        for no in range(1, 11):
            dfs.append(pd.DataFrame(user_data["true_data"][f"test_{no}"]["key_events"]))
            dfs.append(pd.DataFrame(user_data["false_data"][f"test_{no}"]["key_events"]))

big_keyboard_df = pd.concat(dfs).astype({
    "Epoch": "Float64"
})

display(big_keyboard_df.describe(include="all"))
display(big_keyboard_df.isna().sum())

,Key,Event,Input Box,Text Changed,Timestamp,Epoch
count,172563,172563,172563,172563,172563,172563.0
unique,82,2,6,2,171463,<NA>
top,backspace,pressed,Name,True,2022-03-01 13:48:50.519671,<NA>
freq,7925,87608,79905,162836,4,<NA>
mean,NaN,NaN,NaN,NaN,NaN,1647302420.65725
std,NaN,NaN,NaN,NaN,NaN,1935228.31744
min,NaN,NaN,NaN,NaN,NaN,1643113826.341281
25%,NaN,NaN,NaN,NaN,NaN,1646151579.757441
50%,NaN,NaN,NaN,NaN,NaN,1647353218.034949
75%,NaN,NaN,NaN,NaN,NaN,1648984188.621257


Key             0
Event           0
Input Box       0
Text Changed    0
Timestamp       0
Epoch           0
dtype: int64

In [132]:
big_keyboard_df["Key"].unique()

<ArrowStringArray>
[          'shift',               'm',               'r',               's',
        'spacebar',               'w',               'e',       'backspace',
               'l',               'i',             'tab',               '4',
               '2',               '8',               '9',               '1',
               '5',               '0',               '6',               '7',
               '3',          'rshift',         'numpad4',         'numpad2',
         'numpad8',         'numpad9',         'numpad1',         'numpad5',
         'numpad0',         'numpad6',         'numpad7',         'numpad3',
        'capslock', 'numpadsubstract',               '.',               'j',
               ',',               'h',               'd',               't',
               'a',               'y',               'u',               'o',
         'numlock',               'f',           'enter',               'n',
               'g',               '-',               '`',

In [133]:
big_keyboard_df["Event"].value_counts()

Event
pressed     87608
released    84955
Name: count, dtype: int64

The above result suggests that not all key events are paired which implies the dataset has missing events.
Let's look a little deeper into that.

In [188]:
big_keyboard_df.groupby(["Key", "Event"]).count()

Input Box  Text Changed  Timestamp  Epoch
Key Event                                              
#   pressed          22            22         22     22
    released         22            22         22     22
'   pressed           1             1          1      1
    released          1             1          1      1
,   pressed          11            11         11     11
...                 ...           ...        ...    ...
x   released        106           106        106    106
y   pressed         613           613        613    613
    released        613           613        613    613
z   pressed          66            66         66     66
    released         66            66         66     66

[164 rows x 4 columns]

There are small anomalies in shortly pressed keys like `2` and `8` and much bigger discrepancies for `backspace`, `shift`, `rshift`, `spacebar` and more. In all of these instances there are more `pressed` events than `released`, which suggests that a long press may be logged as consecutive `pressed` events.

However, something particularly insteresting happens for `Tab` key with an overwhelming predominance of `released` events.

Let's look into that next.

In [193]:
shifted_df = big_keyboard_df.groupby("Key", sort=False)[["Key","Event", "Epoch"]].shift(-1)
shifted_df.columns = ["Key_shifted", "Event_shifted", "Epoch_shifted"]
comparison_df = pd.concat([big_keyboard_df, shifted_df], axis=1)
comparison_df.loc[(comparison_df["Key"] == "2") & (comparison_df["Event"] == comparison_df["Event_shifted"])]
comparison_df[comparison_df["Epoch"].between(1648896881.0545242 - 5, 1648896881.0545242 + 50)]

,Key,Event,Input Box,Text Changed,Timestamp,Epoch,Key_shifted,Event_shifted,Epoch_shifted
70,4,released,Card No,True,2022-04-02 10:54:36.271858,1648896876.271858,4,pressed,1649073768.692647
71,5,pressed,Card No,True,2022-04-02 10:54:36.418063,1648896876.418063,5,released,1648896876.577644
72,5,released,Card No,True,2022-04-02 10:54:36.577644,1648896876.577644,5,pressed,1648896885.870337
73,7,pressed,Card No,True,2022-04-02 10:54:37.261808,1648896877.261808,7,released,1648896877.469558
74,7,released,Card No,True,2022-04-02 10:54:37.469557,1648896877.469558,7,pressed,1648896897.060272
...,...,...,...,...,...,...,...,...,...
48,0,pressed,Card No,True,2022-04-02 10:55:28.019475,1648896928.020456,0,released,1648896928.16507
49,0,released,Card No,True,2022-04-02 10:55:28.165070,1648896928.16507,0,pressed,1648896943.760422
50,4,pressed,Card No,True,2022-04-02 10:55:29.468384,1648896929.468384,4,released,1648896929.647921
51,4,released,Card No,True,2022-04-02 10:55:29.647920,1648896929.647921,4,pressed,1648896930.985716


In [135]:
big_keyboard_df.loc[big_keyboard_df["Key"] == "shift"]

,Key,Event,Input Box,Text Changed,Timestamp,Epoch
0,shift,pressed,Name,True,2022-02-06 18:57:09.981740,1644173829.9817405
2,shift,released,Name,True,2022-02-06 18:57:10.169470,1644173830.1694703
9,shift,pressed,Name,True,2022-02-06 18:57:10.679335,1644173830.6793358
15,shift,released,Name,True,2022-02-06 18:57:10.844363,1644173830.844363
20,shift,pressed,Name,True,2022-02-06 18:57:11.721636,1644173831.7216363
...,...,...,...,...,...,...
26,shift,pressed,Name,True,2022-04-14 09:59:00.636272,1649930340.6362724
27,shift,pressed,Name,True,2022-04-14 09:59:00.661282,1649930340.6612825
28,shift,pressed,Name,True,2022-04-14 09:59:00.674279,1649930340.6742795
29,shift,pressed,Name,True,2022-04-14 09:59:00.699285,1649930340.6992855


In [136]:
big_keyboard_df.loc[big_keyboard_df["Key"] == "tab"]

,Key,Event,Input Box,Text Changed,Timestamp,Epoch
64,tab,released,Card No,True,2022-02-06 18:57:14.693870,1644173834.6938705
101,tab,released,CVC,True,2022-02-06 18:57:24.125922,1644173844.1259224
108,tab,released,Exp m,True,2022-02-06 18:57:25.602162,1644173845.6021621
113,tab,released,Exp y,True,2022-02-06 18:57:26.707177,1644173846.7071774
54,tab,released,Card No,True,2022-02-06 18:57:33.026100,1644173853.0261004
...,...,...,...,...,...,...
32,tab,released,Card No,True,2022-04-05 09:42:20.966870,1649151740.9668705
65,tab,released,CVC,True,2022-04-05 09:42:45.964544,1649151765.964545
72,tab,released,Exp m,True,2022-04-05 09:42:53.534079,1649151773.5340798
77,tab,released,Exp y,True,2022-04-05 09:42:57.165202,1649151777.1652024


Quick look at `shift` key logs strengthens the suggestion that long presses may be logged as consecutive `pressed` events.

For the `tab` key, I cannot find any reasonable explaination just by looking at the data.

In [194]:
user1_mouse = pd.DataFrame(user1["false_data"]["test_1"]["mouse_events"]).astype({
    "Movement ID": "Int64",
    "Timestamp": "datetime64[us]"
})

display(user1_mouse)
display(user1_mouse.describe(include='all'))
display(user1_mouse.dtypes)

,Event,Coordinates,Timestamp,Epoch,Movement ID,false_enters
0,movement,"[386, 91]",2022-02-06 19:25:28.359473,1644175528.359473,1,NaN
1,left release,"[386, 91.00000000000001]",2022-02-06 19:25:28.408747,1644175528.4087472,<NA>,NaN
2,movement,"[386, 92]",2022-02-06 19:25:28.421404,1644175528.421404,1,NaN
3,movement,"[387, 92]",2022-02-06 19:25:28.471485,1644175528.4714851,1,NaN
4,movement,"[387, 93]",2022-02-06 19:25:28.496569,1644175528.4965694,1,NaN
...,...,...,...,...,...,...
456,movement,"[373, 115]",2022-02-06 19:25:57.417238,1644175557.4172385,11,NaN
457,movement,"[373, 113]",2022-02-06 19:25:57.443276,1644175557.4432764,11,NaN
458,movement,"[374, 113]",2022-02-06 19:25:57.468304,1644175557.4683044,11,NaN
459,left press,"[374, 113.00000000000001]",2022-02-06 19:25:57.517562,1644175557.5175626,<NA>,NaN


,Event,Coordinates,Timestamp,Epoch,Movement ID,false_enters
count,460,460,460,460,446.0,1.0
unique,3,449,NaN,456,<NA>,NaN
top,movement,"[208, 94]",NaN,1644175532.0216825,<NA>,NaN
freq,446,4,NaN,2,<NA>,NaN
mean,NaN,NaN,2022-02-06 19:25:41.622957,NaN,4.553812,0.0
min,NaN,NaN,2022-02-06 19:25:28.359473,NaN,1.0,0.0
25%,NaN,NaN,2022-02-06 19:25:31.510268,NaN,1.0,0.0
50%,NaN,NaN,2022-02-06 19:25:38.197915,NaN,2.0,0.0
75%,NaN,NaN,2022-02-06 19:25:52.593991,NaN,9.0,0.0
max,NaN,NaN,2022-02-06 19:25:57.517562,NaN,11.0,0.0


Event                      str
Coordinates             object
Timestamp       datetime64[us]
Epoch                      str
Movement ID              Int64
false_enters           float64
dtype: object

In [138]:
dfs = []
for file in json_file_list:
    with open(raw_kmt_dataset / file, 'r') as f:
        user_data = json.load(f)
        for no in range(1, 11):
            dfs.append(pd.DataFrame(user_data["true_data"][f"test_{no}"]["mouse_events"]))
            dfs.append(pd.DataFrame(user_data["false_data"][f"test_{no}"]["mouse_events"]))

big_mouse_df = pd.concat(dfs)

display(big_mouse_df.isna().sum())

Event             1760
Coordinates       1760
Timestamp         1760
Epoch             1760
Movement ID      25320
false_enters    706984
dtype: int64

In [139]:
big_mouse_df.loc[big_mouse_df["Event"].isna()].isna().sum()

Event           1760
Coordinates     1760
Timestamp       1760
Epoch           1760
Movement ID     1760
false_enters       0
dtype: int64

In [140]:
big_mouse_df.loc[big_mouse_df["Movement ID"].isna()]

,Event,Coordinates,Timestamp,Epoch,Movement ID,false_enters
55,left press,"[307, 383.00000000000006]",2022-02-06 18:57:08.782240,1644173828.7822409,NaN,NaN
56,left release,"[307, 383.00000000000006]",2022-02-06 18:57:08.856246,1644173828.8562467,NaN,NaN
74,left press,"[318, 312]",2022-02-06 18:57:09.122411,1644173829.1224115,NaN,NaN
78,left release,"[319, 305]",2022-02-06 18:57:09.186461,1644173829.186461,NaN,NaN
83,left press,"[324, 291]",2022-02-06 18:57:09.428796,1644173829.428797,NaN,NaN
...,...,...,...,...,...,...
160,left release,"[422.0, 380.99999999999994]",2022-04-05 09:41:55.375268,1649151715.375268,NaN,NaN
191,left press,"[288.0, 285.0]",2022-04-05 09:41:58.584616,1649151718.5846167,NaN,NaN
192,left release,"[288.0, 285.0]",2022-04-05 09:41:58.812721,1649151718.8127215,NaN,NaN
294,left press,"[376.0, 88.99999999999999]",2022-04-05 09:43:08.412875,1649151788.4128754,NaN,NaN


In [141]:
big_mouse_df.drop(["false_enters"], axis=1, inplace=True)
big_mouse_df.dropna(subset=["Event"], inplace=True)

In [142]:
big_mouse_df.isna().sum()

Event              0
Coordinates        0
Timestamp          0
Epoch              0
Movement ID    23560
dtype: int64

In [143]:
big_mouse_df["Event"].unique()

<ArrowStringArray>
[           'movement',          'left press',        'left release',
    'scrolldown press',  'scrolldown release',      'scrollup press',
    'scrollup release',   'scrollright press', 'scrollright release',
         'right press',       'right release',    'scrollleft press',
  'scrollleft release',        'mouse4 press',      'mouse4 release']
Length: 15, dtype: str

In [144]:
movement_counts = []

for file in json_file_list:
    with open(raw_kmt_dataset / file, 'r') as f:
        user_data = json.load(f)
        for test_no in range(1, 11):
            true_df = pd.DataFrame(user_data["true_data"][f"test_{test_no}"]["mouse_events"])
            movement_counts.append(true_df["Movement ID"].nunique())

            false_df = pd.DataFrame(user_data["false_data"][f"test_{test_no}"]["mouse_events"])
            movement_counts.append(false_df["Movement ID"].nunique())

movement_counts_series = pd.Series(movement_counts)
movement_counts_series.agg([np.mean, np.std, np.min, np.max, np.median])

mean       7.007955
std        5.108810
min        1.000000
max       73.000000
median     7.000000
dtype: float64

In [145]:
mouse_data = {}

for user_file in json_file_list:
    with open(raw_kmt_dataset / user_file, 'r') as f:
        user_data = json.load(f)
        for test_no in range(1, 11):
            mouse_true = user_data["true_data"][f"test_{test_no}"]["mouse_events"]
            for event_id, event in enumerate(mouse_true):
                mouse_data[(int(user_file.split("_")[-1].split(".")[0]), True, test_no, event_id)] = event 

            mouse_false = user_data["false_data"][f"test_{test_no}"]["mouse_events"]
            for event_id, event in enumerate(mouse_false):
                mouse_data[(int(user_file.split("_")[-1].split(".")[0]), False, test_no, event_id)] = event


mouse_df = pd.DataFrame.from_dict(mouse_data, orient='index')
mouse_df.index.names = [
    "user_id",
    "is_genuine",
    "test_id",
    "event_id",
]
mouse_df.set_index(["Event", "Movement ID"], append=True, inplace=True)
mouse_df.drop(["false_enters", "Timestamp"], axis=1, inplace=True)

In [146]:
mouse_df.head()

Coordinates  \
user_id is_genuine test_id event_id Event    Movement ID               
1       True       1       0        movement 1.0           [30, 581]   
                           1        movement 1.0           [76, 569]   
                           2        movement 1.0          [159, 549]   
                           3        movement 1.0          [194, 540]   
                           4        movement 1.0          [239, 531]   

                                                                       Epoch  
user_id is_genuine test_id event_id Event    Movement ID                      
1       True       1       0        movement 1.0          1644173827.5681436  
                           1        movement 1.0          1644173827.5801554  
                           2        movement 1.0          1644173827.5921662  
                           3        movement 1.0          1644173827.6051784  
                           4        movement 1.0          1644173827.6181898

In [147]:
mouse_df.dropna(subset=["Coordinates"], inplace=True)
mouse_df[["X", "Y"]] = mouse_df["Coordinates"].to_list()

mouse_df.drop_duplicates(subset="Epoch", inplace=True)

mouse_df["dX"] = mouse_df["X"].diff()
mouse_df["dY"] = mouse_df["Y"].diff()

mouse_df["Epoch"] = mouse_df["Epoch"].astype('float')
mouse_df["dT"] = mouse_df["Epoch"].diff() * 1_000

mouse_df["dS"] = np.hypot(mouse_df["dX"], mouse_df["dY"])
mouse_df["v"] = mouse_df["dS"] / mouse_df["dT"]
mouse_df["a"] = mouse_df["v"].diff() / mouse_df["dT"]
mouse_df["dvx"] = mouse_df["dX"] / mouse_df["dT"]
mouse_df["dvy"] = mouse_df["dY"] / mouse_df["dT"]
mouse_df["v_hypot"] = np.hypot(mouse_df["dvx"], mouse_df["dvy"])
mouse_df["jerk"] = mouse_df["a"].diff() / mouse_df["dT"]

In [195]:
mouse_df

Coordinates  \
user_id is_genuine test_id event_id Event      Movement ID                               
1       True       1       0        movement   1.0                           [30, 581]   
                           1        movement   1.0                           [76, 569]   
                           2        movement   1.0                          [159, 549]   
                           3        movement   1.0                          [194, 540]   
                           4        movement   1.0                          [239, 531]   
...                                                                                ...   
88      False      10      290      movement   5.0                       [376.0, 88.0]   
                           291      movement   5.0                       [376.0, 87.0]   
                           292      movement   5.0                       [376.0, 88.0]   
                           293      movement   5.0                       [376.0, 89.0]   
                           294      left press NaN          [376.0, 88.99999999999999]   

                                                                   Epoch  \
user_id is_genuine test_id event_id Event      Movement ID                 
1       True       1       0        movement   1.0          1.644174e+09   
                           1        movement   1.0          1.644174e+09   
                           2        movement   1.0          1.644174e+09   
                           3        movement   1.0          1.644174e+09   
                           4        movement   1.0          1.644174e+09   
...                                                                  ...   
88      False      10      290      movement   5.0          1.649152e+09   
                           291      movement   5.0          1.649152e+09   
                           292      movement   5.0          1.649152e+09   
                           293      movement   5.0          1.649152e+09   
                           294      left press NaN          1.649152e+09   

                                                                X      Y  \
user_id is_genuine test_id event_id Event      Movement ID                 
1       True       1       0        movement   1.0           30.0  581.0   
                           1        movement   1.0           76.0  569.0   
                           2        movement   1.0          159.0  549.0   
                           3        movement   1.0          194.0  540.0   
                           4        movement   1.0          239.0  531.0   
...                                                           ...    ...   
88      False      10      290      movement   5.0          376.0   88.0   
                           291      movement   5.0          376.0   87.0   
                           292      movement   5.0          376.0   88.0   
                           293      movement   5.0          376.0   89.0   
                           294      left press NaN          376.0   89.0   

                                                              dX  \
user_id is_genuine test_id event_id Event      Movement ID         
1       True       1       0        movement   1.0           NaN   
                           1        movement   1.0          46.0   
                           2        movement   1.0          83.0   
                           3        movement   1.0          35.0   
                           4        movement   1.0          45.0   
...                                                          ...   
88      False      10      290      movement   5.0           0.0   
                           291      movement   5.0           0.0   
                           292      movement   5.0           0.0   
                           293      movement   5.0           0.0   
                           294      left press NaN           0.0   

                                         

In [149]:
aggregated_mouse = mouse_df.groupby(level=["user_id", "is_genuine", "test_id"]).agg({
    'dS': np.sum,
    'v': [np.mean, np.std, np.median, np.max],
    'a': [np.min, np.max],
    'jerk': np.max
})

In [150]:
aggregated_mouse

dS         v                      \
                                    sum      mean       std    median   
user_id is_genuine test_id                                              
1       False      1        3123.797278  0.503854  1.053869  0.230551   
                   2        2000.718833  0.275405  0.270825  0.185637   
                   3        2338.303267  0.444871  0.506800  0.255946   
                   4        1510.082117  0.371154  0.463199  0.198411   
                   5        2209.246126  0.494828  0.752323  0.245837   
...                                 ...       ...       ...       ...   
88      True       6        1366.152688  0.519610  0.677145  0.263458   
                   7        1067.348284  0.703321  0.908363  0.249914   
                   8         868.697689  0.401736  0.495882  0.263367   
                   9         774.402624  0.445295  0.595913  0.235654   
                   10        821.569933  0.521810  0.681986  0.235715   

                                              a                jerk  
                                  max       min       max       max  
user_id is_genuine test_id                                           
1       False      1        11.866601 -0.468104  0.408248  0.046339  
                   2         1.745583 -0.080999  0.058945  0.008543  
                   3         3.511004 -0.143088  0.125916  0.018906  
                   4         3.405164 -0.249191  2.728044  2.641612  
                   5         8.270517 -0.648206  7.458422  7.258357  
...                               ...       ...       ...       ...  
88      True       6         4.240926 -0.362158  3.804349  3.841818  
                   7         3.994367 -0.204576  0.284355  0.024524  
                   8         3.300208 -0.176014  0.271013  0.023264  
                   9         3.353200 -0.096867  0.180543  0.022709  
                   10        3.410138 -0.142677  2.059275  2.070180  

[1760 rows x 8 columns]

In [151]:
from sklearn.naive_bayes import GaussianNB
from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn import metrics
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [152]:
X = aggregated_mouse.loc[(1)]
y = X.index.get_level_values("is_genuine").astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3, stratify=y, random_state=42)


clf_tree = tree.DecisionTreeClassifier()

clf_tree.fit(X_train, y_train)
clf_tree.score(X_test, y_test)


1.0

In [153]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

clf = DecisionTreeClassifier(random_state=42, )

scores = cross_val_score(
    clf,
    X,
    y,
    cv=cv,
)

scores.mean(), scores.std()

(np.float64(0.9), np.float64(0.1224744871391589))

In [155]:
from xgboost.sklearn import XGBClassifier
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(
    XGBClassifier(n_jobs=-1),
    param_grid={
        "max_depth": [3, 5, 7],
        "learning_rate": [0.001, .01, .05],
    },
    cv=4,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=3
)

search.fit(X_train, y_train)

clf = search.best_estimator_


cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42,
)


scores = cross_val_score(
    clf,
    X,
    y,
    cv=cv,
    verbose=3,
    n_jobs=-1
)

scores.mean(), scores.std(), scores, clf

Fitting 4 folds for each of 9 candidates, totalling 36 fits
[CV 1/4] END ..learning_rate=0.001, max_depth=3;, score=0.750 total time=   0.1s
[CV 2/4] END ..learning_rate=0.001, max_depth=3;, score=0.750 total time=   0.1s
[CV 3/4] END ..learning_rate=0.001, max_depth=3;, score=1.000 total time=   0.1s
[CV 3/4] END ..learning_rate=0.001, max_depth=7;, score=1.000 total time=   0.0s
[CV 4/4] END ..learning_rate=0.001, max_depth=3;, score=1.000 total time=   0.1s
[CV 3/4] END ..learning_rate=0.001, max_depth=5;, score=1.000 total time=   0.1s
[CV 4/4] END ..learning_rate=0.001, max_depth=7;, score=1.000 total time=   0.0s
[CV 1/4] END ...learning_rate=0.01, max_depth=3;, score=0.750 total time=   0.0s
[CV 2/4] END ..learning_rate=0.001, max_depth=5;, score=0.750 total time=   0.1s
[CV 2/4] END ...learning_rate=0.01, max_depth=3;, score=0.750 total time=   0.0s
[CV 1/4] END ..learning_rate=0.001, max_depth=5;, score=0.750 total time=   0.1s
[CV 4/4] END ..learning_rate=0.001, max_depth=5;,

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of   4 | elapsed:    0.0s finished


(np.float64(0.4),
 np.float64(0.0),
 array([0.4, 0.4, 0.4, 0.4]),
 XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=True, eval_metric=None, feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=0.001, max_bin=None, max_cat_threshold=None,
               max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
               max_leaves=None, min_child_weight=None, missing=nan,
               monotone_constraints=None, multi_strategy=None, n_estimators=None,
               n_jobs=-1, num_parallel_tree=None, ...))

In [156]:
clf

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None
